# Small selective-SSM language model on Kaggle

This notebook is deliberately opt-in: it validates an attached/prepared token dataset and runs a small smoke test before training. It never silently downloads FineWeb or starts a long run. Set `RUN_DATA_PREPARE` and `RUN_TRAIN` explicitly after reviewing the token budget and dataset terms.

Attach this repository (or a Kaggle Dataset containing it) and a separate tokenized dataset if the data was prepared in an earlier session. Save checkpoints from `/kaggle/working/outputs` as notebook output or publish them as a versioned Kaggle Dataset for later sessions.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, torch, yaml

# Change this only if the project is mounted under a different Kaggle input name.
candidates = [Path('/kaggle/working/sslm'), Path('/kaggle/input/sslm-bundle'), Path('/kaggle/input/sslm'), Path('/kaggle/input/sslm-bundle/sslm_bundle')]
PROJECT = next((p for p in candidates if (p / 'src').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('Attach the project as a Kaggle Dataset at /kaggle/input/sslm or copy it to /kaggle/working/sslm')
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print('Project:', PROJECT)
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.version.cuda)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

In [ ]:
# Kaggle normally already provides CUDA PyTorch. Install only project-side packages.
%pip install -q numpy==1.26.4 PyYAML==6.0.2 tokenizers==0.19.1 tqdm==4.66.5 datasets==2.19.2 pytest==8.3.5

## Explicit run controls

`base_50m.yaml` expects a token manifest at `data/tokenized/general/tokens` and a tokenizer beside it. For a first GPU check, use `configs/tiny.yaml` with a small prepared fixture or change `CONFIG` to a prepared dataset configuration.

In [ ]:
CONFIG = 'configs/base_50m.yaml'  # 'configs/tiny.yaml' for a small run
RUN_DATA_PREPARE = False          # True only after setting a reviewed token budget
RUN_TRAIN = False                 # True starts the configured run
RUN_EVAL = False
TOKEN_BUDGET = 100_000_000

# Keep Kaggle's preserved output separate from source/input datasets.
KAGGLE_OUTPUT = Path('/kaggle/working/outputs')
KAGGLE_OUTPUT.mkdir(parents=True, exist_ok=True)
run_cfg = yaml.safe_load(Path(CONFIG).read_text())
run_cfg['output_dir'] = str(KAGGLE_OUTPUT / 'checkpoints')
RUN_CONFIG = Path('/kaggle/working/sslm_run_config.yaml')
RUN_CONFIG.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
print({'config': CONFIG, 'run_config': RUN_CONFIG, 'token_budget': TOKEN_BUDGET, 'run_data_prepare': RUN_DATA_PREPARE, 'run_train': RUN_TRAIN})

In [ ]:
if RUN_DATA_PREPARE:
    env = os.environ.copy()
    env['TOKEN_BUDGET'] = str(TOKEN_BUDGET)
    env['RAW_ROOT'] = '/kaggle/working/data/raw/general'
    env['PROCESSED_ROOT'] = '/kaggle/working/data/processed/general'
    env['TOKEN_ROOT'] = '/kaggle/working/data/tokenized/general'
    subprocess.run(['bash', 'scripts/prepare_data.sh'], check=True, env=env)
    run_cfg['data']['token_manifest'] = '/kaggle/working/data/tokenized/general/tokens'
    run_cfg['generation']['tokenizer'] = '/kaggle/working/data/tokenized/general/tokenizer.json'
    RUN_CONFIG.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
else:
    print('Data preparation skipped; using already attached tokenized data.')

In [ ]:
# Validate the manifest before allocating a model.
import yaml
cfg = run_cfg
manifest_path = Path(cfg['data']['token_manifest']) / 'manifest.json'
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    print(json.dumps({'vocab_size': manifest.get('vocab_size'), 'total_tokens': manifest.get('total_tokens'), 'splits': {k: len(v) for k, v in manifest.get('splits', {}).items()}}, indent=2))
else:
    print('Token manifest is not present yet:', manifest_path)

In [ ]:
# Always run the local architecture/data test suite and the forward/backward smoke test.
subprocess.run(['make', 'test'], check=True)
subprocess.run([sys.executable, '-m', 'src.training.train', '--config', 'configs/tiny.yaml', '--dry-run'], check=True)

In [ ]:
if RUN_TRAIN:
    # To preserve results, point output_dir in the selected YAML at /kaggle/working/outputs/...
    subprocess.run([sys.executable, '-m', 'src.training.train', '--config', str(RUN_CONFIG)], check=True)
else:
    print('Training skipped. Set RUN_TRAIN=True only after checking CONFIG and token budget.')

In [ ]:
if RUN_EVAL:
    cfg = run_cfg
    checkpoint_dir = Path(cfg.get('output_dir', 'outputs/checkpoints'))
    checkpoint = checkpoint_dir / 'best.pt'
    subprocess.run([sys.executable, '-m', 'src.training.evaluate', '--checkpoint', str(checkpoint), '--token-manifest', cfg['data']['token_manifest'], '--max-batches', '50', '--output', str(KAGGLE_OUTPUT / 'evaluation.json')], check=True)
else:
    print('Evaluation skipped.')

## Persistence reminder

Kaggle `/kaggle/working` is preserved only when the notebook version is saved. Before ending a session, save the notebook output or publish `outputs/checkpoints` plus the tokenizer as a new Kaggle Dataset. A fresh session cannot resume from a path that was not attached or saved.